# Simple RAG (Retrieval-Augmented Generation) Implementation with LangChain 1.0+

## Overview
This notebook demonstrates a complete RAG pipeline using **LangChain 1.0+ with LCEL** (LangChain Expression Language).

### What is RAG?
RAG combines retrieval of relevant documents with generation from a Large Language Model (LLM):
1. **Retrieval**: Find relevant information from a knowledge base
2. **Augmentation**: Add retrieved context to the prompt
3. **Generation**: LLM generates answers based on the context

### Pipeline Flow:
```
PDF Documents → Load → Split into Chunks → Create Embeddings → Store in Vector DB
                                                                         ↓
User Query → Retrieve Similar Chunks → Combine with Query → LLM → Answer
```

### Components Used:
- **Document Loader**: PyPDFLoader (for PDF processing)
- **Text Splitter**: RecursiveCharacterTextSplitter (smart chunking)
- **Embeddings**: OpenAI text-embedding-3-small (vector representations)
- **Vector Store**: FAISS (fast similarity search)
- **LLM**: OpenAI GPT-4-Turbo or GPT-3.5-Turbo
- **Chain Builder**: LCEL (LangChain Expression Language)

### LangChain 1.0+ Features:
- ✅ Modern LCEL syntax with pipe operator `|`
- ✅ More readable and composable chains
- ✅ Better streaming support
- ✅ Type-safe operations

---

In [2]:
# Check installed package versions
import sys
from importlib.metadata import version

try:
    import langchain
    print(f"✓ langchain: {langchain.__version__}")
except:
    print("✗ langchain not installed")

try:
    import langchain_core
    print(f"✓ langchain-core: {langchain_core.__version__}")
except:
    print("✗ langchain-core not installed - REQUIRED!")
    print("  Run: pip install langchain-core")

try:
    import langchain_openai
    print(f"✓ langchain-openai: {version('langchain-openai')}")
except:
    print("✗ langchain-openai not installed")

try:
    import langchain_community
    print(f"✓ langchain-community: {langchain_community.__version__}")
except:
    print("✗ langchain-community not installed")

print(f"\nPython version: {sys.version}")
print("\nIf any packages are missing, run:")
print("uv pip install langchain langchain-core langchain-openai langchain-community langchain-text-splitters faiss-cpu pypdf python-dotenv tiktoken")

✓ langchain: 1.2.14
✓ langchain-core: 1.2.24
✓ langchain-openai: 1.1.12
✓ langchain-community: 0.4.1

Python version: 3.12.12 (main, Oct 28 2025, 11:52:25) [Clang 20.1.4 ]

If any packages are missing, run:
uv pip install langchain langchain-core langchain-openai langchain-community langchain-text-splitters faiss-cpu pypdf python-dotenv tiktoken


In [4]:
# Standard library imports
import os

#os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
from pathlib import Path

# Environment variable management - for secure API key handling
from dotenv import load_dotenv

# LangChain Document Loaders - for loading PDF documents
from langchain_community.document_loaders import PyPDFLoader

# LangChain Text Splitters - for breaking documents into manageable chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# OpenAI Integration - for embeddings and LLM
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# Vector Store - FAISS for efficient similarity search
from langchain_community.vectorstores import FAISS

# LangChain Core Components
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("✓ All imports successful!")
print("✓ Compatible with LangChain 1.0+")

✓ All imports successful!
✓ Compatible with LangChain 1.0+


## Environment Configuration

### Setting up OpenAI API Key

You have two options:
1. **Recommended**: Create a `.env` file with `OPENAI_API_KEY=your_key_here`
2. **Alternative**: Set it directly in code (not recommended for production)

Get your API key from: https://platform.openai.com/api-keys

In [5]:
# Load environment variables from .env file
load_dotenv()

# Verify API key is loaded
if not os.getenv("OPENAI_API_KEY"):
    print("⚠️  WARNING: OPENAI_API_KEY not found!")
    print("Please set it in .env file or uncomment the line below:")
    # os.environ["OPENAI_API_KEY"] = "your_api_key_here"
else:
    print("✓ OpenAI API Key loaded successfully!")
    print(f"✓ Key starts with: {os.getenv('OPENAI_API_KEY')[:8]}...")

✓ OpenAI API Key loaded successfully!
✓ Key starts with: sk-proj-...


## 4. Document Loading

### Loading PDF Documents

PyPDFLoader extracts text from PDF files page by page. Each page becomes a separate document with metadata (page number, source file).

**How it works:**
- Reads PDF files and extracts text content
- Preserves page numbers for source tracking
- Returns Document objects with `.page_content` and `.metadata`

**Note**: Update the `pdf_path` variable to point to your PDF file(s).

In [6]:
# ===== CONFIGURATION: Update this path to your PDF file =====
pdf_path = "pdfs/rag.pdf"  # Change this to your PDF file path
# =============================================================

# Check if file exists
if not os.path.exists(pdf_path):
    print(f"⚠️  ERROR: File '{pdf_path}' not found!")
    print("Please update the pdf_path variable with your PDF file location.")
else:
    # Initialize the PDF loader
    loader = PyPDFLoader(pdf_path)
    
    # Load all pages from the PDF
    # Each page becomes a separate Document object
    documents = loader.load()
    
    # Display information about loaded documents
    print(f"✓ Loaded {len(documents)} pages from '{pdf_path}'")
    print(f"\n--- First Document Preview ---")
    print(f"Content (first 500 chars): {documents[0].page_content[:500]}...")
    print(f"\nMetadata: {documents[0].metadata}")
    print(f"\nTotal characters across all pages: {sum(len(doc.page_content) for doc in documents):,}")

✓ Loaded 19 pages from 'pdfs/rag.pdf'

--- First Document Preview ---
Content (first 500 chars): Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†
†Facebook AI Research;‡University College London;⋆New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge
in their parameters, and achieve state-of-the-art results when ﬁ...

Metadata: {'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'pdfs/rag.pdf', 'total_pages': 19, 'page'

### Loading Multiple PDFs

If you have multiple PDF files, you can load them all at once:

In [7]:
# Example: Loading multiple PDFs from a directory
# Uncomment and modify if you want to load multiple files

pdf_directory = "./pdfs"  # Directory containing your PDFs
all_documents = []

if os.path.exists(pdf_directory):
    pdf_files = list(Path(pdf_directory).glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files")
    
    for pdf_file in pdf_files:
        loader = PyPDFLoader(str(pdf_file))
        docs = loader.load()
        all_documents.extend(docs)
        print(f"  ✓ Loaded {len(docs)} pages from {pdf_file.name}")
    
    print(f"\nTotal pages loaded: {len(all_documents)}")
    documents = all_documents  # Use this for the rest of the pipeline

Found 4 PDF files
  ✓ Loaded 19 pages from rag.pdf
  ✓ Loaded 33 pages from Understanding_Climate_Change.pdf
  ✓ Loaded 15 pages from attention_paper.pdf
  ✓ Loaded 21 pages from ragsurvey.pdf

Total pages loaded: 88


## 5. Text Splitting

### Why Split Documents?
- LLMs have token limits (e.g., 4K, 8K, 128K tokens)
- Smaller chunks = more precise retrieval
- Balance: chunks must be large enough to contain meaningful context but small enough to be specific

### RecursiveCharacterTextSplitter
This splitter tries to keep related text together by recursively splitting on:
1. Paragraphs (`\n\n`)
2. Lines (`\n`)
3. Sentences (`. `)
4. Words (` `)
5. Characters (as last resort)

**Parameters:**
- `chunk_size=1024`: Target size for each chunk (in characters)
- `chunk_overlap=128`: Overlap between chunks to maintain context continuity
- Overlap prevents important information from being split across chunks

In [8]:
# Initialize the text splitter with recommended settings
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024,        # Maximum characters per chunk (roughly 200-250 tokens)
    chunk_overlap=128,      # Characters overlap between chunks (maintains context)
    length_function=len,    # Function to measure chunk length
    separators=["\n\n", "\n", " ", ""]  # Try to split on paragraphs first, then lines, etc.
)

# Split the documents into chunks
# This creates smaller, manageable pieces while preserving semantic meaning
chunks = text_splitter.split_documents(documents)

# Display splitting results
print(f"✓ Split {len(documents)} documents into {len(chunks)} chunks")
print(f"\nAverage chunk size: {sum(len(chunk.page_content) for chunk in chunks) / len(chunks):.0f} characters")

# Preview a few chunks
print(f"\n--- Chunk Examples ---")
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i+1} (length: {len(chunk.page_content)} chars):")
    print(f"{chunk.page_content[:200]}...")
    print(f"Metadata: {chunk.metadata}")

✓ Split 88 documents into 363 chunks

Average chunk size: 875 characters

--- Chunk Examples ---

Chunk 1 (length: 1006 chars):
Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, W...
Metadata: {'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'pdfs/rag.pdf', 'total_pages': 19, 'page': 0, 'page_label': '1'}

Chunk 2 (length: 1023 chars):
memory have so far been only investigated for extractive downstream tasks. We
explore a general-purpose ﬁne-tuning recipe for retrieval-augmented generation
(RAG) — models which combine pre-trained pa...
Metadata: {

## 6. Creating Embeddings

### What are Embeddings?
Embeddings are vector representations of text that capture semantic meaning. Similar texts have similar vectors.

**Example**: 
- "dog" and "puppy" → similar vectors (close in vector space)
- "dog" and "spaceship" → different vectors (far apart)

### OpenAI text-embedding-3-small
- **Dimensions**: 1536 (each text becomes a 1536-dimensional vector)
- **Cost**: $0.00002 per 1,000 tokens (very affordable)
- **Performance**: 62.3% on MTEB benchmark
- **Speed**: Fast and efficient

**Alternative**: `text-embedding-3-large` for higher quality (64.6% MTEB) at higher cost

In [9]:
# Initialize OpenAI Embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",  # Latest, cost-effective embedding model
    # dimensions=1536
    # Alternative: "text-embedding-3-large" for better quality
)

# Test the embeddings with a sample text
sample_text = "This is a test sentence to demonstrate embeddings."
sample_embedding = embeddings.embed_query(sample_text)

print(f"✓ Embeddings model initialized: text-embedding-3-small")
print(f"✓ Embedding dimension: {len(sample_embedding)}")
print(f"✓ Sample embedding (first 10 values): {sample_embedding[:10]}")
print(f"\nℹ️  Each chunk will be converted to a {len(sample_embedding)}-dimensional vector for similarity search")

✓ Embeddings model initialized: text-embedding-3-small
✓ Embedding dimension: 1536
✓ Sample embedding (first 10 values): [0.0203704833984375, -0.003147125244140625, -0.0005393028259277344, 0.004558563232421875, -0.0149688720703125, -0.034027099609375, 0.01763916015625, 0.019622802734375, 0.0013074874877929688, 0.005908966064453125]

ℹ️  Each chunk will be converted to a 1536-dimensional vector for similarity search


## 7. Creating Vector Store (FAISS)

### What is a Vector Store?
A vector store (or vector database) stores embeddings and enables fast similarity search.

### FAISS (Facebook AI Similarity Search)
- **Fast**: Optimized for billion-scale vector search
- **Local**: Runs on your machine, no cloud dependency
- **Efficient**: Uses advanced indexing algorithms

### How Similarity Search Works:
1. Convert query to embedding vector
2. Find vectors in the database most similar to query vector (using cosine similarity or Euclidean distance)
3. Return the corresponding text chunks

**This cell will:**
1. Convert all chunks to embeddings (may take a minute for large documents)
2. Build a FAISS index
3. Save to disk for future use

In [10]:
# Create FAISS vector store from document chunks
# This step converts each chunk to an embedding and stores it
print(f"Creating FAISS index from {len(chunks)} chunks...")
print("This may take a minute depending on the number of chunks...")

vectorstore = FAISS.from_documents(
    documents=chunks,      # Our split document chunks
    embedding=embeddings   # OpenAI embedding model
)

print(f"✓ FAISS vector store created successfully!")
print(f"✓ Indexed {len(chunks)} document chunks")

# Save the vector store to disk for later use
# This allows you to reload the index without re-processing documents
vectorstore_path = "./faiss_index"
vectorstore.save_local(vectorstore_path)
print(f"✓ Vector store saved to '{vectorstore_path}'")
print(f"\nℹ️  You can reload this index later using: FAISS.load_local('{vectorstore_path}', embeddings)")

Creating FAISS index from 363 chunks...
This may take a minute depending on the number of chunks...
✓ FAISS vector store created successfully!
✓ Indexed 363 document chunks
✓ Vector store saved to './faiss_index'

ℹ️  You can reload this index later using: FAISS.load_local('./faiss_index', embeddings)


In [11]:
# Create a retriever from the vector store
retriever = vectorstore.as_retriever(
    search_type="similarity",    # Use cosine similarity for search
    search_kwargs={"k": 4}        # Retrieve top 4 most relevant chunks
)

print("✓ Retriever configured successfully")
print(f"  - Search type: similarity")
print(f"  - Number of documents to retrieve (k): 4")

# Test the retriever with a sample query
# Note: In LangChain 1.0+, use .invoke() instead of .get_relevant_documents()
test_query = "What is Open-domain Question Answering?"
retrieved_docs = retriever.invoke(test_query)  # LangChain 1.0+ method

print(f"\n--- Retriever Test ---")
print(f"Query: '{test_query}'")
print(f"Retrieved {len(retrieved_docs)} documents:")

for i, doc in enumerate(retrieved_docs):
    print(f"\nDocument {i+1}:")
    print(f"  Content preview: {doc.page_content[:150]}...")
    print(f"  Metadata: {doc.metadata}")

✓ Retriever configured successfully
  - Search type: similarity
  - Number of documents to retrieve (k): 4

--- Retriever Test ---
Query: 'What is Open-domain Question Answering?'
Retrieved 4 documents:

Document 1:
  Content preview: D Further Details on Open-Domain QA
For open-domain QA, multiple answer annotations are often available for a given question. These
answer annotations...
  Metadata: {'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'pdfs/rag.pdf', 'total_pages': 19, 'page': 17, 'page_label': '18'}

Document 2:
  Content preview: Open-domain question answering (QA) is an important real-world application and common testbed
for knowledge-intensive tasks [20]. We treat questions a...

In [ ]:
# Initialize the ChatOpenAI model
llm = ChatOpenAI(
      model="gpt-5.4-mini",  
      temperature=0,         # 0 = deterministic, factual responses (recommended for Q&A)
      max_tokens=2000,       # Maximum length of response
  )

print("✓ LLM configured successfully")
print(f"  - Model: gpt-4-turbo-2024-04-09")
print(f"  - Temperature: 0 (deterministic)")
print(f"  - Max tokens: 2000")

# Test the LLM with a simple query
test_response = llm.invoke("Say 'Hello, I am ready to answer questions!'")
print(f"\nLLM Test Response: {test_response.content}")


✓ LLM configured successfully
  - Model: gpt-4-turbo-2024-04-09
  - Temperature: 0 (deterministic)
  - Max tokens: 2000

LLM Test Response: Hello, I am ready to answer questions!


## 8. Creating the RAG Chain (LangChain 1.0+ LCEL)

### What is a RAG Chain?
The RAG chain combines retrieval and generation into a single workflow:
1. User asks a question
2. Retriever finds relevant documents
3. Documents are formatted as context
4. LLM generates answer using the context

### LangChain 1.0+ LCEL (LangChain Expression Language)
LangChain 1.0+ uses LCEL, a declarative way to build chains using the pipe operator `|`.

**Benefits:**
- More intuitive and readable
- Better streaming support
- Easier to debug and modify
- Type-safe and composable

**Components:**
- **RunnablePassthrough**: Passes input through unchanged
- **Pipe operator (|)**: Chains components together
- **StrOutputParser**: Converts LLM output to string

In [13]:
# Define the prompt template for the RAG system
# This tells the LLM how to use the retrieved context
system_prompt = (
    "You are a helpful assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer based on the context, say that you don't know. "
    "Keep the answer concise and accurate.\n\n"
    "Context: {context}\n\n"
    "Question: {question}"
)

# Create the prompt template
prompt = ChatPromptTemplate.from_template(system_prompt)

# Helper function to format documents
def format_docs(docs):
    """Format retrieved documents into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain using LangChain 1.0+ LCEL (LangChain Expression Language)
# This uses the pipe operator (|) to chain components together
rag_chain = (
    {
        "context": retriever | format_docs,  # Retrieve docs and format them
        "question": RunnablePassthrough()      # Pass through the question
    }
    | prompt           # Format with prompt template
    | llm              # Generate answer with LLM
    | StrOutputParser() # Parse output to string
)

print("✓ RAG chain created successfully using LangChain 1.0+ LCEL!")
print("\nRAG Pipeline Flow:")
print("  1. User provides a query")
print("  2. Retriever finds top 4 relevant chunks")
print("  3. Chunks are formatted as context")
print("  4. Context + question are formatted with prompt template")
print("  5. LLM generates answer based on context")
print("  6. Answer is parsed and returned to user")

✓ RAG chain created successfully using LangChain 1.0+ LCEL!

RAG Pipeline Flow:
  1. User provides a query
  2. Retriever finds top 4 relevant chunks
  3. Chunks are formatted as context
  4. Context + question are formatted with prompt template
  5. LLM generates answer based on context
  6. Answer is parsed and returned to user


In [14]:
# Example Query 1: General question about the document
query1 = "What is Open-domain Question Answering?"

print(f"Query: {query1}")
print("\nProcessing...\n")

# With LangChain 1.0+, we invoke the chain with the question directly
answer = rag_chain.invoke(query1)

print("=" * 80)
print("ANSWER:")
print("=" * 80)
print(answer)
print("\n" + "=" * 80)

# To see which documents were retrieved, we can call the retriever separately
print("\nSOURCE DOCUMENTS USED:")
print("=" * 80)
retrieved_docs = retriever.invoke(query1)
for i, doc in enumerate(retrieved_docs):
    print(f"\nDocument {i+1}:")
    print(f"  Source: {doc.metadata}")
    print(f"  Content: {doc.page_content[:200]}...")
    print("-" * 80)

Query: What is Open-domain Question Answering?

Processing...

ANSWER:
Open-domain question answering is a knowledge-intensive QA task where a model answers questions using a large external knowledge source, often by retrieving documents and generating or extracting the answer.


SOURCE DOCUMENTS USED:

Document 1:
  Source: {'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'pdfs/rag.pdf', 'total_pages': 19, 'page': 17, 'page_label': '18'}
  Content: D Further Details on Open-Domain QA
For open-domain QA, multiple answer annotations are often available for a given question. These
answer annotations are exploited by extractive models during trainin...
--------------------------------------------

In [15]:
# Example Query 2: Specific information extraction
query2 = "Whta is non-parametric memory in RAG?"

print(f"Query: {query2}")
print("\nProcessing...\n")

answer = rag_chain.invoke(query2)

print("=" * 80)
print("ANSWER:")
print("=" * 80)
print(answer)
print("\n" + "=" * 80)

Query: Whta is non-parametric memory in RAG?

Processing...

ANSWER:
In RAG, **non-parametric memory** is the **dense vector index of Wikipedia passages** that the model retrieves from using a neural retriever. It is **not trainable parameters** of the generator, but an external stored memory of **21M 728-dimensional vectors** that can be searched during generation.



In [16]:
# Example Query 3: Your custom question
# Replace this with your own question!
custom_query = "What specific details are mentioned about attention mechanisms?"

print(f"Query: {custom_query}")
print("\nProcessing...\n")

answer = rag_chain.invoke(custom_query)

print("=" * 80)
print("ANSWER:")
print("=" * 80)
print(answer)
print("\n" + "=" * 80)

Query: What specific details are mentioned about attention mechanisms?

Processing...

ANSWER:
The context says attention mechanisms in the Transformer are **self-attention** (also called **intra-attention**), which relates different positions within a sequence to build its representation. It also mentions that:

- **Multi-Head Attention** is used to counteract averaging effects.
- Attention can capture **long-distance dependencies**.
- Some heads appear to help with **anaphora resolution**.
- The Transformer relies **entirely on self-attention** rather than recurrence or convolutions.



In [17]:
# Example Query 3: Your custom question
# Replace this with your own question!
custom_query = "What are the applications of Attention Mechanism?"

print(f"Query: {custom_query}")
print("\nProcessing...\n")

response3 = rag_chain.invoke(custom_query)

print("=" * 80)
print("ANSWER:")
print("=" * 80)
print(response3)
print("\n" + "=" * 80)

Query: What are the applications of Attention Mechanism?

Processing...

ANSWER:
The context says attention mechanisms have been used successfully in tasks like:

- reading comprehension
- abstractive summarization
- textual entailment
- learning task-independent sentence representations
- simple-language question answering
- language modeling

It also mentions they help model dependencies between input and output sequences.

